# Education defense check — does the 2001 religiosity dating survive?

**Verdict: ARMY — SURVIVES.** nr × WV4 = +0.716*** → **+0.636** (p = .001) with education × wave modeled jointly: −11%, well inside the ~30% band, and most of that is sample composition, not education (same-sample baseline = +0.694, so the education adjustment itself costs 8%).
**POLICE — unchanged** (nr × WV4 +0.45 → +0.50). **COURTS — contrast holds** (no WV4 item; all nr × wave n.s. in both models).

**Question.** The moderator scan (`results/moderator_scan_grid.csv`) found education × army interactions significant in all waves including WV3 (+0.65), predating the WV4 religiosity split. Education and religiosity are correlated cleavages — does the headline `nr × WV4 = +0.716***` (layer4b, WV2 baseline) survive when education realignment enters the same model?

**Design.** (1) Exact layer4b replication gate; (2) joint model = layer4b spec + `ed_high` (education_recode = 3, as in the scan) main effect + ed_high × wave block — nested, nothing else changes; (3) side-by-side nr × wave comparison incl. a baseline-on-joint-sample column to separate sample composition from covariate adjustment; (4) collinearity check; (5) same loop for police and courts.

Defensive/exploratory check — no existing files modified, main models untouched. New output: `results/education_defense_table.csv`.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
from statsmodels.miscmodels.ordinal_model import OrderedModel

warnings.filterwarnings("ignore")
RESULTS = "results"
LATER = ["WV3", "WV4", "WV5", "WV6", "WV7"]
INST = ["conf_military", "conf_police", "conf_courts"]

df = pd.read_csv(os.path.join(RESULTS, "pooled_extended_v2.csv"))
df = df.dropna(subset=["cluster"]).copy()          # layer4b analysis sample
df["not_religious_dummy"] = df["rel_person"].map(
    lambda v: 1 if v in (2, 3) else (0 if v == 1 else np.nan))
df["ed_high"] = np.where(df["education_recode"].isna(), np.nan,
                         (df["education_recode"] == 3).astype(float))
for w in LATER:
    df[f"wave_{w}"] = (df["wave"] == w).astype(int)
    df[f"nr_x_{w}"] = df["not_religious_dummy"] * df[f"wave_{w}"]
    df[f"ed_x_{w}"] = df["ed_high"] * df[f"wave_{w}"]

BASE = ["not_religious_dummy", "rel_attendance", "rel_importance_god",
        "left_right", "age", "female", "education_raw", "income_raw",
        "married_dummy"] + [f"wave_{w}" for w in LATER] + [f"nr_x_{w}" for w in LATER]
ED_BLOCK = ["ed_high"] + [f"ed_x_{w}" for w in LATER]

def fit(outcome, preds, frame):
    sub = frame[[outcome] + preds].dropna().copy()
    sub[outcome] = sub[outcome].astype(int)
    keep = [p for p in preds if sub[p].nunique() > 1]
    res = OrderedModel(sub[outcome], sub[keep], distr="logit").fit(
        method="bfgs", maxiter=400, disp=False)
    assert res.mle_retvals.get("converged", False), f"non-convergence: {outcome}"
    return res, len(sub)

print(f"Analysis sample: n = {len(df)}")

Analysis sample: n = 11311


## 1. Replication gate

Exact layer4b army specification. Coefficients are asserted (tolerance 1e-3) against the published `results/layer4b_longwindow_interactions.csv` — if this cell fails, everything downstream is void.

In [2]:
res_rep, n_rep = fit("conf_military", BASE, df)
pub = pd.read_csv(os.path.join(RESULTS, "layer4b_longwindow_interactions.csv"))
pub = pub[(pub.institution == "conf_military")
          & pub.predictor.str.startswith("nr_x_")].set_index("predictor")
check = pd.DataFrame({
    "replicated": [res_rep.params[t] for t in pub.index],
    "published":  pub["coef"].values}, index=pub.index).round(4)
assert n_rep == int(pub["n_obs"].iloc[0]), "sample size mismatch vs layer4b"
assert np.allclose(check["replicated"], check["published"], atol=1e-3), \
    "replication FAILED — stop"
print(f"REPLICATED: n = {n_rep}, nr_x_WV4 = {res_rep.params['nr_x_WV4']:+.4f} "
      f"(SE {res_rep.bse['nr_x_WV4']:.4f}, p = {res_rep.pvalues['nr_x_WV4']:.5f})")
check

REPLICATED: n = 8915, nr_x_WV4 = +0.7162 (SE 0.1828, p = 0.00009)


,replicated,published
predictor,,
nr_x_WV3,0.3746,0.3746
nr_x_WV4,0.7162,0.7162
nr_x_WV5,0.6352,0.6352
nr_x_WV6,0.4023,0.4023
nr_x_WV7,0.6511,0.6511


## 2. Joint models, all three institutions

Per institution, three fits: **baseline** (layer4b spec, full sample), **baseline│joint sample** (same spec on education-nonmissing rows, ~755 fewer — isolates sample composition), **joint** (+ ed_high + ed × wave). `education_raw` stays in as the level control throughout, exactly as in layer4b; the only change in the joint model is the added realignment block. Courts has no WV4 cell (item not asked).

In [3]:
rows = []
for inst in INST:
    joint_frame = df.dropna(subset=["ed_high"])
    rb,  n_b  = fit(inst, BASE, df)
    rbs, n_bs = fit(inst, BASE, joint_frame)
    rj,  n_j  = fit(inst, BASE + ED_BLOCK, df)
    assert n_bs == n_j
    for w in LATER:
        nr, ed = f"nr_x_{w}", f"ed_x_{w}"
        if nr not in rb.params.index:
            continue
        rows.append({
            "institution": inst, "wave": w,
            "nr_base_coef": rb.params[nr],  "nr_base_se": rb.bse[nr],
            "nr_base_p": rb.pvalues[nr],
            "nr_base_jsample_coef": rbs.params[nr], "nr_base_jsample_p": rbs.pvalues[nr],
            "nr_joint_coef": rj.params[nr], "nr_joint_se": rj.bse[nr],
            "nr_joint_p": rj.pvalues[nr],
            "ed_joint_coef": rj.params[ed], "ed_joint_se": rj.bse[ed],
            "ed_joint_p": rj.pvalues[ed],
            "n_base": n_b, "n_joint": n_j})
    print(f"  {inst}: baseline n={n_b}, joint n={n_j}")

table = pd.DataFrame(rows).round(4)
table.to_csv(os.path.join(RESULTS, "education_defense_table.csv"), index=False)
print("\nSaved results/education_defense_table.csv")

  conf_military: baseline n=8915, joint n=8160


  conf_police: baseline n=8911, joint n=8157


  conf_courts: baseline n=6231, joint n=5743

Saved results/education_defense_table.csv


In [4]:
def stars(p):
    return "***" if p < .001 else "** " if p < .01 else "*  " if p < .05 else "   "

for inst in INST:
    sub = table[table.institution == inst]
    print(f"\n=== {inst} ===  (baseline n={sub.n_base.iloc[0]}, joint n={sub.n_joint.iloc[0]})")
    print(f"{'wave':5s} {'nr baseline':>19s} {'nr base|jsample':>19s} "
          f"{'nr joint':>19s} {'ed joint':>19s}")
    for _, r in sub.iterrows():
        print(f"{r.wave:5s}"
              f" {r.nr_base_coef:+.3f}{stars(r.nr_base_p)} p={r.nr_base_p:.4f}"
              f" {r.nr_base_jsample_coef:+.3f}{stars(r.nr_base_jsample_p)} p={r.nr_base_jsample_p:.4f}"
              f" {r.nr_joint_coef:+.3f}{stars(r.nr_joint_p)} p={r.nr_joint_p:.4f}"
              f" {r.ed_joint_coef:+.3f}{stars(r.ed_joint_p)} p={r.ed_joint_p:.4f}")

k = table[(table.institution == "conf_military") & (table.wave == "WV4")].iloc[0]
drop_total = 100 * (1 - k.nr_joint_coef / k.nr_base_coef)
drop_covar = 100 * (1 - k.nr_joint_coef / k.nr_base_jsample_coef)
verdict = ("SURVIVES" if k.nr_joint_p < .05 and drop_total < 30 else
           "ATTENUATED" if k.nr_joint_p < .05 and drop_total < 60 else "ABSORBED")
print(f"\nTHE NUMBER: army nr × WV4 joint = {k.nr_joint_coef:+.4f} "
      f"(p = {k.nr_joint_p:.4f}); change vs +0.716: −{drop_total:.1f}% total, "
      f"−{drop_covar:.1f}% net of sample composition → {verdict}")


=== conf_military ===  (baseline n=8915, joint n=8160)
wave          nr baseline     nr base|jsample            nr joint            ed joint
WV3   +0.375    p=0.0607 +0.409    p=0.0530 +0.366    p=0.0884 +0.608**  p=0.0096
WV4   +0.716*** p=0.0001 +0.694*** p=0.0003 +0.636**  p=0.0010 +0.673**  p=0.0015
WV5   +0.635**  p=0.0045 +0.573*   p=0.0163 +0.501*   p=0.0400 +0.760**  p=0.0014
WV6   +0.402    p=0.0509 +0.457*   p=0.0345 +0.332    p=0.1325 +0.979*** p=0.0000
WV7   +0.651*** p=0.0003 +0.653*** p=0.0003 +0.571**  p=0.0020 +0.805*** p=0.0002

=== conf_police ===  (baseline n=8911, joint n=8157)
wave          nr baseline     nr base|jsample            nr joint            ed joint
WV3   +0.282    p=0.1382 +0.310    p=0.1205 +0.305    p=0.1317 +0.225    p=0.3155
WV4   +0.448*   p=0.0107 +0.449*   p=0.0129 +0.498**  p=0.0067 -0.062    p=0.7630
WV5   +0.261    p=0.2058 +0.216    p=0.3261 +0.219    p=0.3289 +0.187    p=0.4131
WV6   +0.155    p=0.4394 +0.194    p=0.3557 +0.101    p=0.6374

## 3. Interpretation

The 2001 dating **SURVIVES**: army nr × WV4 moves from +0.716*** to +0.636** (p = .001) when education realignment is modeled jointly — an 11% drop, far inside the 30% band, and the same-sample baseline (+0.694) shows most of it is who gets dropped for missing education, not the education adjustment (−8%). Education × wave has real independent structure in the army model (all waves positive, peaking at WV6 = +0.98), but it sits *alongside* the religiosity signal rather than absorbing it — the only casualty is nr × WV6, which slips from marginal (p = .051) to n.s. (p = .13), and WV6 is already the flagged anomaly wave (handoff finding B); the dating claim rests on WV4 and is untouched. The institution-specific contrast also holds in the joint model: police nr × wave is essentially unchanged (nr × WV4 +0.45 → +0.50), and courts religiosity interactions stay flat (all n.s., two mildly negative), so army-specific dissolution + courts stability survives intact.

## 4. Collinearity sanity check

How much overlap is the joint model fighting? If nr and ed_high were near-duplicates, a "SURVIVES" verdict would be meaningless (the model couldn't separate them and would split the effect arbitrarily).

In [5]:
cc = df[["not_religious_dummy", "ed_high"]].dropna()
print(f"corr(not_religious, ed_high), pooled: {cc.corr().iloc[0, 1]:+.3f}  (n={len(cc)})")

blk = df[["conf_military"] + BASE + ED_BLOCK].dropna()
X = blk[["not_religious_dummy", "ed_high"]
        + [f"nr_x_{w}" for w in LATER] + [f"ed_x_{w}" for w in LATER]]
print("\nsame-wave corr(nr × w, ed × w):",
      {w: round(X.corr().loc[f"nr_x_{w}", f"ed_x_{w}"], 3) for w in LATER})

Xs = ((X - X.mean()) / X.std()).values
print(f"condition number (standardized block): {np.linalg.cond(Xs):.1f}  "
      f"(< 30 = benign)")
vif = pd.Series(np.diag(np.linalg.inv(X.corr().values)), index=X.columns).round(2)
print("\nVIFs (main effects are inflated by their own interaction terms — expected):")
print(vif.to_string())

corr(not_religious, ed_high), pooled: +0.114  (n=9391)

same-wave corr(nr × w, ed × w): {'WV3': 0.358, 'WV4': 0.286, 'WV5': 0.385, 'WV6': 0.377, 'WV7': 0.232}
condition number (standardized block): 9.0  (< 30 = benign)

VIFs (main effects are inflated by their own interaction terms — expected):
not_religious_dummy     7.91
ed_high                16.43
nr_x_WV3                2.53
nr_x_WV4                3.35
nr_x_WV5                2.01
nr_x_WV6                2.15
nr_x_WV7                3.79
ed_x_WV3                4.55
ed_x_WV4                6.94
ed_x_WV5                4.38
ed_x_WV6                5.80
ed_x_WV7                4.08


**Read:** overlap is modest — pooled corr +0.11, same-wave interaction correlations 0.23–0.39, condition number ≈ 9, interaction-term VIFs ≤ 7. The design can separate the two cleavages; the verdict is substantive, not a collinearity artifact. (WV7 education cells carry the ISCED-recode caveat noted in the scan; the verdict does not depend on them.)